In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rl-deep-rl",
    notebook_name="02_dqn_from_scratch_experiments.ipynb"
)

# Experiments: DQN From Scratch

This notebook produces runnable evidence for the three key claims about DQN. Each experiment isolates one component — experience replay, target network, or epsilon schedule — and measures what happens when you remove or change it. Every output here is something you can point to in an interview.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
import matplotlib.pyplot as plt
import copy

print("Imports ready.")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

---
## Shared Components: Environment, Network, Buffer, Training Loop

All three experiments use the same self-contained environment and agent. No external dependencies (no gym, no RL libraries).

### Chain MDP

A 10-state chain. The agent starts at state 0 and wants to reach state 9.

```
  State:   0 --- 1 --- 2 --- 3 --- 4 --- 5 --- 6 --- 7 --- 8 --- 9
           ← left                                            right →
           
  Actions: 0 = left, 1 = right
  Reward:  +10 at state 9 (terminal), -1 every other step
  γ = 0.99
  Max steps per episode: 50
```

States are encoded as one-hot vectors (10-dimensional). This keeps things simple and self-contained while still requiring a neural network to generalize across states.

In [ ]:
# ============================================================
# Chain MDP — 10-state environment
# ============================================================

class ChainMDP:
    """
    10-state chain MDP.
    
    States 0-9, actions: left (0), right (1).
    Reward +10 at state 9 (terminal), -1 each step.
    States encoded as one-hot vectors.
    """
    def __init__(self, n_states=10, max_steps=50):
        self.n_states = n_states
        self.n_actions = 2
        self.max_steps = max_steps
        self.state = 0
        self.steps = 0
    
    def reset(self):
        self.state = 0
        self.steps = 0
        return self._one_hot(self.state)
    
    def _one_hot(self, s):
        vec = np.zeros(self.n_states, dtype=np.float32)
        vec[s] = 1.0
        return vec
    
    def step(self, action):
        self.steps += 1
        
        if action == 1:  # right
            self.state = min(self.state + 1, self.n_states - 1)
        else:  # left
            self.state = max(self.state - 1, 0)
        
        # Terminal state: state 9
        if self.state == self.n_states - 1:
            return self._one_hot(self.state), 10.0, True
        
        # Timeout
        if self.steps >= self.max_steps:
            return self._one_hot(self.state), -1.0, True
        
        return self._one_hot(self.state), -1.0, False


# ============================================================
# Q-Network — small 2-layer network
# ============================================================

class QNetwork(nn.Module):
    """Small Q-network: state (10) -> hidden (64) -> hidden (64) -> actions (2)."""
    def __init__(self, state_dim=10, action_dim=2, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
    
    def forward(self, x):
        return self.net(x)


# ============================================================
# Replay Buffer
# ============================================================

class ReplayBuffer:
    """Fixed-size FIFO replay buffer with random sampling."""
    def __init__(self, capacity=1000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)),
            torch.LongTensor(actions),
            torch.FloatTensor(rewards),
            torch.FloatTensor(np.array(next_states)),
            torch.FloatTensor(dones)
        )
    
    def __len__(self):
        return len(self.buffer)


# ============================================================
# DQN Training Loop (configurable)
# ============================================================

def train_dqn(
    n_episodes=500,
    use_replay=True,
    replay_capacity=1000,
    batch_size=32,
    use_target_net=True,
    target_update_steps=100,
    epsilon_schedule="linear_decay",  # "fixed", "linear_decay", "exponential_decay"
    epsilon_fixed=None,
    gamma=0.99,
    lr=1e-3,
    seed=42,
    track_q_state0=True,
    track_q_all_states=False,
):
    """
    Train a DQN agent on the 10-state chain MDP.
    
    Returns a dict with training history:
      - 'rewards': episode rewards
      - 'q_state0': Q-value of state 0 (best action) at each episode end
      - 'mean_q_steps': mean Q across all states at each training step (if track_q_all_states)
    """
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    
    env = ChainMDP()
    q_net = QNetwork()
    optimizer = optim.Adam(q_net.parameters(), lr=lr)
    
    # Target network
    if use_target_net:
        target_net = copy.deepcopy(q_net)
    else:
        target_net = None
    
    # Replay buffer
    buffer = ReplayBuffer(capacity=replay_capacity) if use_replay else None
    
    # Epsilon schedule
    def get_epsilon(episode):
        if epsilon_schedule == "fixed":
            return epsilon_fixed if epsilon_fixed is not None else 0.1
        elif epsilon_schedule == "linear_decay":
            return max(0.01, 1.0 - episode / (n_episodes * 0.8))
        elif epsilon_schedule == "exponential_decay":
            return max(0.01, 0.995 ** episode)
        else:
            return 0.1
    
    # Tracking
    rewards_history = []
    q_state0_history = []
    mean_q_step_history = []
    global_step = 0
    
    # One-hot for state 0 (for Q-value tracking)
    state0_tensor = torch.FloatTensor(np.eye(10, dtype=np.float32)[0:1])  # shape (1, 10)
    
    # All states (for Q-value tracking)
    all_states_tensor = torch.FloatTensor(np.eye(10, dtype=np.float32))  # shape (10, 10)
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        epsilon = get_epsilon(episode)
        done = False
        
        while not done:
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = np.random.randint(env.n_actions)
            else:
                with torch.no_grad():
                    q_vals = q_net(torch.FloatTensor(state).unsqueeze(0))
                    action = q_vals.argmax(dim=1).item()
            
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # --- Training ---
            if use_replay:
                buffer.push(state, action, reward, next_state, float(done))
                
                if len(buffer) >= batch_size:
                    b_states, b_actions, b_rewards, b_next, b_dones = buffer.sample(batch_size)
                    
                    current_q = q_net(b_states).gather(1, b_actions.unsqueeze(1)).squeeze(1)
                    
                    with torch.no_grad():
                        net_for_target = target_net if use_target_net else q_net
                        next_q = net_for_target(b_next).max(dim=1)[0]
                        targets = b_rewards + gamma * next_q * (1 - b_dones)
                    
                    loss = nn.functional.mse_loss(current_q, targets)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
            else:
                # Online: train on each transition immediately
                s_t = torch.FloatTensor(state).unsqueeze(0)
                ns_t = torch.FloatTensor(next_state).unsqueeze(0)
                
                current_q = q_net(s_t)[0, action]
                
                with torch.no_grad():
                    net_for_target = target_net if use_target_net else q_net
                    next_q = net_for_target(ns_t).max(dim=1)[0]
                    target = reward + gamma * next_q * (1 - float(done))
                
                loss = nn.functional.mse_loss(current_q.unsqueeze(0), target)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            # Update target network
            global_step += 1
            if use_target_net and global_step % target_update_steps == 0:
                target_net.load_state_dict(q_net.state_dict())
            
            # Track Q-values per step (for Experiment 2)
            if track_q_all_states:
                with torch.no_grad():
                    all_q = q_net(all_states_tensor)
                    mean_q_step_history.append(all_q.max(dim=1)[0].mean().item())
            
            state = next_state
        
        rewards_history.append(total_reward)
        
        # Track Q(state 0) at end of each episode
        if track_q_state0:
            with torch.no_grad():
                q0 = q_net(state0_tensor).max(dim=1)[0].item()
                q_state0_history.append(q0)
    
    return {
        'rewards': rewards_history,
        'q_state0': q_state0_history,
        'mean_q_steps': mean_q_step_history,
    }


# --- Verify everything works ---
print("SHARED COMPONENTS")
print("=" * 60)

env = ChainMDP()
obs = env.reset()
print(f"ChainMDP: {env.n_states} states, {env.n_actions} actions")
print(f"State 0 (one-hot): {obs}")
print(f"State shape: {obs.shape}")

# Step right 9 times to reach terminal
total_r = 0
for i in range(9):
    obs, r, done = env.step(1)  # right
    total_r += r
print(f"\nAfter 9 right steps: state={np.argmax(obs)}, "
      f"total reward={total_r:.0f}, done={done}")
print(f"(8 steps at -1 each = -8, plus +10 at goal = +2 total)")

q_net = QNetwork()
n_params = sum(p.numel() for p in q_net.parameters())
print(f"\nQNetwork: {n_params:,} parameters")
print(f"Architecture: 10 -> 64 -> 64 -> 2")

# Quick sanity check: train for 5 episodes
result = train_dqn(n_episodes=5, seed=0)
print(f"\nSanity check (5 episodes): rewards = {result['rewards']}")

# Compute analytical optimal Q(s=0)
# Optimal policy: always go right. Takes 9 steps from state 0.
# Reward sequence: -1, -1, -1, -1, -1, -1, -1, -1, +10
# Q*(s=0, right) = sum_{t=0}^{7} gamma^t * (-1) + gamma^8 * 10
gamma = 0.99
q_star_s0 = sum(gamma**t * (-1) for t in range(8)) + gamma**8 * 10
print(f"\nAnalytical Q*(state=0, right) = {q_star_s0:.4f}")
print(f"(This is the target value for Q(state 0) in our experiments)")
print("=" * 60)

---
## Experiment 1: Experience Replay Ablation

**Claim being tested:** "Without replay, DQN overfits to recent experiences and performance is unstable."

**Why it matters in an interview:** Experience replay is one of the two critical innovations in DQN. An interviewer may ask: "What happens if you remove the replay buffer?" You need to show (not just say) that performance degrades and Q-values become noisy.

**What we measure:**
- DQN **with** replay buffer (size 1000, batch 32) vs DQN **without** replay (online training on each transition)
- Both conditions use a target network (to isolate the replay effect)
- 20 seeds, 500 episodes each
- Track: (a) episode rewards, (b) Q-value for state 0

In [ ]:
# --- Experiment 1: Run replay vs no-replay across 20 seeds ---

N_SEEDS = 20
N_EPISODES = 500

print("EXPERIMENT 1: EXPERIENCE REPLAY ABLATION")
print("=" * 60)
print(f"Conditions: with replay (buffer=1000, batch=32) vs without replay (online)")
print(f"Both conditions use a target network (update every 100 steps)")
print(f"Seeds: {N_SEEDS}, Episodes per seed: {N_EPISODES}")
print()

rewards_with_replay = []    # shape will be (N_SEEDS, N_EPISODES)
rewards_no_replay = []
q0_with_replay = []
q0_no_replay = []

for s in range(N_SEEDS):
    # With replay
    result_replay = train_dqn(
        n_episodes=N_EPISODES,
        use_replay=True,
        replay_capacity=1000,
        batch_size=32,
        use_target_net=True,
        target_update_steps=100,
        epsilon_schedule="linear_decay",
        gamma=0.99,
        lr=1e-3,
        seed=s,
        track_q_state0=True,
    )
    rewards_with_replay.append(result_replay['rewards'])
    q0_with_replay.append(result_replay['q_state0'])
    
    # Without replay
    result_no_replay = train_dqn(
        n_episodes=N_EPISODES,
        use_replay=False,
        use_target_net=True,
        target_update_steps=100,
        epsilon_schedule="linear_decay",
        gamma=0.99,
        lr=1e-3,
        seed=s,
        track_q_state0=True,
    )
    rewards_no_replay.append(result_no_replay['rewards'])
    q0_no_replay.append(result_no_replay['q_state0'])
    
    if (s + 1) % 5 == 0:
        print(f"  Completed {s+1}/{N_SEEDS} seeds")

rewards_with_replay = np.array(rewards_with_replay)  # (20, 500)
rewards_no_replay = np.array(rewards_no_replay)
q0_with_replay = np.array(q0_with_replay)
q0_no_replay = np.array(q0_no_replay)

print()
print(f"Final avg reward (last 50 episodes):")
print(f"  With replay:    {rewards_with_replay[:, -50:].mean():.2f} "
      f"(std across seeds: {rewards_with_replay[:, -50:].mean(axis=1).std():.2f})")
print(f"  Without replay: {rewards_no_replay[:, -50:].mean():.2f} "
      f"(std across seeds: {rewards_no_replay[:, -50:].mean(axis=1).std():.2f})")
print()
print(f"Final Q(state 0) (last 50 episodes):")
print(f"  With replay:    {q0_with_replay[:, -50:].mean():.4f}")
print(f"  Without replay: {q0_no_replay[:, -50:].mean():.4f}")
print(f"  Analytical Q*:  {q_star_s0:.4f}")
print("=" * 60)

In [ ]:
# --- Experiment 1: Plot ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Smoothing function
def smooth(data, window=20):
    """Smooth a 1D array with a moving average."""
    return np.convolve(data, np.ones(window) / window, mode='valid')

# --- Left: Smoothed reward curves ---
ax1 = axes[0]
window = 20

# Mean and std across seeds
mean_replay = rewards_with_replay.mean(axis=0)
std_replay = rewards_with_replay.std(axis=0)
mean_no_replay = rewards_no_replay.mean(axis=0)
std_no_replay = rewards_no_replay.std(axis=0)

# Smooth the means
sm_replay = smooth(mean_replay, window)
sm_no_replay = smooth(mean_no_replay, window)
sm_std_replay = smooth(std_replay, window)
sm_std_no_replay = smooth(std_no_replay, window)
x = np.arange(window - 1, N_EPISODES)

ax1.plot(x, sm_replay, 'b-', linewidth=2, label='With Replay')
ax1.fill_between(x, sm_replay - sm_std_replay, sm_replay + sm_std_replay,
                 alpha=0.2, color='blue')
ax1.plot(x, sm_no_replay, 'r-', linewidth=2, label='Without Replay (online)')
ax1.fill_between(x, sm_no_replay - sm_std_no_replay, sm_no_replay + sm_std_no_replay,
                 alpha=0.2, color='red')

ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('Episode Reward', fontsize=12)
ax1.set_title('Reward Curves: Replay vs No Replay', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# --- Right: Q-value evolution for state 0 ---
ax2 = axes[1]

mean_q0_replay = q0_with_replay.mean(axis=0)
std_q0_replay = q0_with_replay.std(axis=0)
mean_q0_no_replay = q0_no_replay.mean(axis=0)
std_q0_no_replay = q0_no_replay.std(axis=0)

sm_q0_replay = smooth(mean_q0_replay, window)
sm_q0_no_replay = smooth(mean_q0_no_replay, window)
sm_q0_std_replay = smooth(std_q0_replay, window)
sm_q0_std_no_replay = smooth(std_q0_no_replay, window)

ax2.plot(x, sm_q0_replay, 'b-', linewidth=2, label='With Replay')
ax2.fill_between(x, sm_q0_replay - sm_q0_std_replay,
                 sm_q0_replay + sm_q0_std_replay, alpha=0.2, color='blue')
ax2.plot(x, sm_q0_no_replay, 'r-', linewidth=2, label='Without Replay')
ax2.fill_between(x, sm_q0_no_replay - sm_q0_std_no_replay,
                 sm_q0_no_replay + sm_q0_std_no_replay, alpha=0.2, color='red')

# Analytical optimal Q-value
gamma = 0.99
q_star_s0 = sum(gamma**t * (-1) for t in range(8)) + gamma**8 * 10
ax2.axhline(y=q_star_s0, color='green', linestyle='--', linewidth=2,
            label=f'Q* = {q_star_s0:.2f} (analytical)')

ax2.set_xlabel('Episode', fontsize=12)
ax2.set_ylabel('Q(state 0, best action)', fontsize=12)
ax2.set_title('Q-Value for State 0: Stability Comparison', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key numbers
print("\nKey numbers for the interview:")
print(f"  Q-value std (last 100 eps), with replay:    {q0_with_replay[:, -100:].std():.4f}")
print(f"  Q-value std (last 100 eps), without replay: {q0_no_replay[:, -100:].std():.4f}")
ratio = q0_no_replay[:, -100:].std() / max(q0_with_replay[:, -100:].std(), 1e-8)
print(f"  Instability ratio (no-replay / replay): {ratio:.1f}x")

**What the output shows:** Without replay, the agent trains on one transition at a time, in sequence. Consecutive transitions come from similar states (e.g., states 3, 4, 5 in a row). This means the network overfits to whatever region of the state space it visited most recently and forgets what it learned about other regions. The reward curve is noisier and the Q-value estimates for state 0 fluctuate more than with replay.

With replay, random batches of 32 transitions from across the entire training history break this correlation. The network sees a diverse mix of states in every update, which leads to smoother learning and more stable Q-value estimates.

**Interview sentence:** "Without experience replay, DQN trains on correlated sequential transitions, which causes the network to overfit to recent states and forget old ones. Replay breaks this correlation by sampling random mini-batches from a buffer, giving more stable Q-values and smoother reward curves."

---
## Experiment 2: Target Network Ablation

**Claim being tested:** "Without a target network, Q-values can grow without bound due to the positive feedback loop."

**Why it matters in an interview:** The target network is DQN's other critical innovation. The interviewer wants to hear you explain the bootstrapping instability: when the same network provides both the prediction and the target, updating the prediction shifts the target, which causes the next update to overshoot, and so on.

**What we measure:**
- DQN **with** target network (update every 100 steps) vs **without** target network (use same network for targets)
- Both conditions use replay buffer (to isolate the target network effect)
- 20 seeds, 500 episodes each
- Track mean Q-values across all 10 states over training steps (not episodes)
- Compare final Q-values to the analytical solution

In [ ]:
# --- Experiment 2: Run target-net vs no-target-net across 20 seeds ---

N_SEEDS_EXP2 = 20
N_EPISODES_EXP2 = 500

print("EXPERIMENT 2: TARGET NETWORK ABLATION")
print("=" * 60)
print(f"Conditions: with target net (update every 100 steps) vs without")
print(f"Both conditions use replay buffer (capacity=1000, batch=32)")
print(f"Seeds: {N_SEEDS_EXP2}, Episodes per seed: {N_EPISODES_EXP2}")
print()

mean_q_with_target = []  # each entry: list of mean-Q per training step
mean_q_no_target = []
final_q_with_target = []  # Q-values for all 10 states at end of training
final_q_no_target = []

for s in range(N_SEEDS_EXP2):
    # With target network
    result_target = train_dqn(
        n_episodes=N_EPISODES_EXP2,
        use_replay=True,
        replay_capacity=1000,
        batch_size=32,
        use_target_net=True,
        target_update_steps=100,
        epsilon_schedule="linear_decay",
        gamma=0.99,
        lr=1e-3,
        seed=s,
        track_q_state0=False,
        track_q_all_states=True,
    )
    mean_q_with_target.append(result_target['mean_q_steps'])
    
    # Without target network
    result_no_target = train_dqn(
        n_episodes=N_EPISODES_EXP2,
        use_replay=True,
        replay_capacity=1000,
        batch_size=32,
        use_target_net=False,
        epsilon_schedule="linear_decay",
        gamma=0.99,
        lr=1e-3,
        seed=s,
        track_q_state0=False,
        track_q_all_states=True,
    )
    mean_q_no_target.append(result_no_target['mean_q_steps'])
    
    if (s + 1) % 5 == 0:
        print(f"  Completed {s+1}/{N_SEEDS_EXP2} seeds")

# Align lengths: different seeds may have slightly different step counts
# (due to episode length variation). Truncate to the shortest run.
min_len_target = min(len(h) for h in mean_q_with_target)
min_len_no_target = min(len(h) for h in mean_q_no_target)
min_len = min(min_len_target, min_len_no_target)

mean_q_with_target = np.array([h[:min_len] for h in mean_q_with_target])  # (20, steps)
mean_q_no_target = np.array([h[:min_len] for h in mean_q_no_target])

print()
print(f"Training steps per run: ~{min_len}")
print(f"\nFinal mean Q (last 200 steps):")
print(f"  With target net:    {mean_q_with_target[:, -200:].mean():.4f} "
      f"(std: {mean_q_with_target[:, -200:].std():.4f})")
print(f"  Without target net: {mean_q_no_target[:, -200:].mean():.4f} "
      f"(std: {mean_q_no_target[:, -200:].std():.4f})")

# Analytical optimal Q* for each state
gamma = 0.99
q_star_all = []
for state_idx in range(9):  # states 0-8
    steps_to_goal = 9 - state_idx - 1
    q_val = sum(gamma**t * (-1) for t in range(steps_to_goal)) + gamma**steps_to_goal * 10
    q_star_all.append(q_val)
q_star_all.append(0.0)  # state 9 is terminal
q_star_mean = np.mean(q_star_all)

print(f"\nAnalytical mean Q* across all states: {q_star_mean:.4f}")
print("=" * 60)

In [ ]:
# --- Experiment 2: Plot ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Q-value trajectories over training steps ---
ax1 = axes[0]
window = 100

avg_target = mean_q_with_target.mean(axis=0)
std_target = mean_q_with_target.std(axis=0)
avg_no_target = mean_q_no_target.mean(axis=0)
std_no_target = mean_q_no_target.std(axis=0)

sm_target = smooth(avg_target, window)
sm_no_target = smooth(avg_no_target, window)
sm_std_t = smooth(std_target, window)
sm_std_nt = smooth(std_no_target, window)
x_steps = np.arange(window - 1, min_len)

ax1.plot(x_steps, sm_target, 'b-', linewidth=2, label='With Target Net')
ax1.fill_between(x_steps, sm_target - sm_std_t, sm_target + sm_std_t,
                 alpha=0.2, color='blue')
ax1.plot(x_steps, sm_no_target, 'r-', linewidth=2, label='Without Target Net')
ax1.fill_between(x_steps, sm_no_target - sm_std_nt, sm_no_target + sm_std_nt,
                 alpha=0.2, color='red')

ax1.axhline(y=q_star_mean, color='green', linestyle='--', linewidth=2,
            label=f'Analytical Q* mean = {q_star_mean:.2f}')

ax1.set_xlabel('Training Step', fontsize=12)
ax1.set_ylabel('Mean Q-value (all states)', fontsize=12)
ax1.set_title('Q-Value Trajectories: Target Net Effect', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- Right: Final Q-value accuracy bar chart ---
ax2 = axes[1]

# Mean absolute error of final Q-values vs analytical
final_q_target_mean = mean_q_with_target[:, -200:].mean()
final_q_no_target_mean = mean_q_no_target[:, -200:].mean()
final_q_target_std = mean_q_with_target[:, -200:].mean(axis=1).std()
final_q_no_target_std = mean_q_no_target[:, -200:].mean(axis=1).std()

mae_target = abs(final_q_target_mean - q_star_mean)
mae_no_target = abs(final_q_no_target_mean - q_star_mean)

labels = ['With\nTarget Net', 'Without\nTarget Net', 'Analytical\nQ*']
values = [final_q_target_mean, final_q_no_target_mean, q_star_mean]
stds = [final_q_target_std, final_q_no_target_std, 0]
colors = ['#42a5f5', '#ef5350', '#66bb6a']

bars = ax2.bar(labels, values, color=colors, edgecolor='black', linewidth=1.5,
               yerr=stds, capsize=8)
ax2.set_ylabel('Mean Q-value (last 200 steps)', fontsize=12)
ax2.set_title('Final Q-Value Accuracy', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("Key numbers:")
print(f"  MAE vs Q* (with target net):    {mae_target:.4f}")
print(f"  MAE vs Q* (without target net): {mae_no_target:.4f}")
print(f"  Q-value std across seeds (with target):    {final_q_target_std:.4f}")
print(f"  Q-value std across seeds (without target): {final_q_no_target_std:.4f}")

**What the output shows:** Without a target network, the Q-network's own predictions are used as targets. Every gradient step changes both the prediction and the target simultaneously. This creates a positive feedback loop: if Q-values are overestimated, the target (which uses the same overestimated network) is also too high, which pushes Q-values even higher. The Q-value trajectory without a target network shows more oscillation and may drift away from the analytical optimum.

With a target network, the targets are frozen for 100 steps at a time. The online network can update freely without shifting its own goalposts. This decouples the prediction from the target, breaking the feedback loop. The Q-values converge more smoothly toward Q*.

**Interview sentence:** "Without a target network, DQN suffers from bootstrapping instability: the network that generates the target is the same one being updated, creating a positive feedback loop. The target network breaks this loop by providing a frozen target that only updates every C steps, leading to more stable convergence."

---
## Experiment 3: Epsilon Schedule Benchmark

**Claim being tested:** "Epsilon-greedy with decay balances exploration (finding reward) and exploitation (maximizing reward)."

**Why it matters in an interview:** Exploration strategy is a fundamental design choice in RL. The interviewer may ask: "How do you set epsilon?" or "What happens with too much or too little exploration?" You need to show that the extremes (always random, always greedy) fail, and a decaying schedule works best.

**What we measure:**
- Full DQN (with replay + target net) under 4 epsilon schedules:
  - (a) Fixed epsilon = 1.0 (always random)
  - (b) Fixed epsilon = 0.0 (always greedy from the start)
  - (c) Linear decay: 1.0 -> 0.01 over 80% of training
  - (d) Exponential decay: epsilon = 0.995^episode
- 10 seeds, 500 episodes each

In [ ]:
# --- Experiment 3: Run 4 epsilon schedules across 10 seeds ---

N_SEEDS_EXP3 = 10
N_EPISODES_EXP3 = 500

print("EXPERIMENT 3: EPSILON SCHEDULE BENCHMARK")
print("=" * 60)
print("Schedules:")
print("  (a) Fixed eps=1.0 (always random)")
print("  (b) Fixed eps=0.0 (always greedy)")
print("  (c) Linear decay: 1.0 -> 0.01")
print("  (d) Exponential decay: 0.995^episode")
print(f"All use full DQN (replay + target net)")
print(f"Seeds: {N_SEEDS_EXP3}, Episodes per seed: {N_EPISODES_EXP3}")
print()

schedules = {
    'always_random': {'epsilon_schedule': 'fixed', 'epsilon_fixed': 1.0},
    'always_greedy': {'epsilon_schedule': 'fixed', 'epsilon_fixed': 0.0},
    'linear_decay':  {'epsilon_schedule': 'linear_decay'},
    'exp_decay':     {'epsilon_schedule': 'exponential_decay'},
}

schedule_labels = {
    'always_random': '(a) Fixed \u03b5=1.0 (random)',
    'always_greedy': '(b) Fixed \u03b5=0.0 (greedy)',
    'linear_decay':  '(c) Linear decay',
    'exp_decay':     '(d) Exponential decay',
}

all_rewards = {}  # {name: (N_SEEDS, N_EPISODES)}

for name, params in schedules.items():
    seed_rewards = []
    for s in range(N_SEEDS_EXP3):
        result = train_dqn(
            n_episodes=N_EPISODES_EXP3,
            use_replay=True,
            replay_capacity=1000,
            batch_size=32,
            use_target_net=True,
            target_update_steps=100,
            gamma=0.99,
            lr=1e-3,
            seed=s,
            track_q_state0=False,
            track_q_all_states=False,
            **params
        )
        seed_rewards.append(result['rewards'])
    
    all_rewards[name] = np.array(seed_rewards)
    final_mean = all_rewards[name][:, -50:].mean()
    print(f"  {schedule_labels[name]:35s} | Final avg reward (last 50): {final_mean:.2f}")

print()
print("=" * 60)

In [ ]:
# --- Experiment 3: Plot ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Reward curves for all 4 schedules ---
ax1 = axes[0]
window = 20

colors_map = {
    'always_random': '#ef5350',
    'always_greedy': '#ffa726',
    'linear_decay':  '#42a5f5',
    'exp_decay':     '#66bb6a',
}

for name in schedules:
    data = all_rewards[name]
    mean_r = data.mean(axis=0)
    std_r = data.std(axis=0)
    sm_mean = smooth(mean_r, window)
    sm_std = smooth(std_r, window)
    x_eps = np.arange(window - 1, N_EPISODES_EXP3)
    
    ax1.plot(x_eps, sm_mean, color=colors_map[name], linewidth=2,
             label=schedule_labels[name])
    ax1.fill_between(x_eps, sm_mean - sm_std, sm_mean + sm_std,
                     alpha=0.15, color=colors_map[name])

ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('Episode Reward', fontsize=12)
ax1.set_title('Reward Curves: 4 Epsilon Schedules', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9, loc='lower right')
ax1.grid(True, alpha=0.3)

# --- Right: Final mean reward bar chart with std error ---
ax2 = axes[1]

bar_names = list(schedules.keys())
bar_labels = [schedule_labels[n] for n in bar_names]
# Mean across last 50 episodes, then mean and std across seeds
bar_means = [all_rewards[n][:, -50:].mean(axis=1).mean() for n in bar_names]
bar_stds = [all_rewards[n][:, -50:].mean(axis=1).std() / np.sqrt(N_SEEDS_EXP3)
            for n in bar_names]  # standard error
bar_colors = [colors_map[n] for n in bar_names]

x_pos = np.arange(len(bar_names))
bars = ax2.bar(x_pos, bar_means, color=bar_colors, edgecolor='black', linewidth=1.5,
               yerr=bar_stds, capsize=8)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(['Always\nrandom', 'Always\ngreedy', 'Linear\ndecay', 'Exp\ndecay'],
                     fontsize=10)
ax2.set_ylabel('Mean Reward (last 50 eps)', fontsize=12)
ax2.set_title('Final Performance by Schedule', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, bar_means):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary
print("\nFinal performance summary (mean +/- std error, last 50 episodes):")
for name in bar_names:
    per_seed_means = all_rewards[name][:, -50:].mean(axis=1)
    m = per_seed_means.mean()
    se = per_seed_means.std() / np.sqrt(N_SEEDS_EXP3)
    print(f"  {schedule_labels[name]:35s} | {m:6.2f} +/- {se:.2f}")

**What the output shows:** The two extremes both fail:

- **Always random (epsilon=1.0):** The agent finds the goal sometimes by accident, but never exploits what it learns. It takes random walks forever. Performance stays low.
- **Always greedy (epsilon=0.0):** The agent gets stuck with its initial (random) Q-values. If the first few random weights happen to favor going left, the agent never explores right, never finds the goal, and never learns. Performance is inconsistent across seeds.

The decay schedules work because they start with high exploration (finding the goal) and gradually shift to exploitation (reaching it efficiently).

- **Linear decay** reduces epsilon steadily to 0.01, giving a smooth transition.
- **Exponential decay** reduces epsilon quickly at first, then slowly. It can converge faster but may under-explore in early training.

**Interview sentence:** "Exploration and exploitation are in tension. With epsilon=1.0 the agent explores but never exploits; with epsilon=0.0 it exploits random initial Q-values and may never find the reward. A decaying schedule starts with exploration to discover the reward structure, then shifts to exploitation to maximize it. Linear decay is simple and robust; exponential decay converges faster but may miss states that need more exploration."

---
## Summary

Claims now backed by evidence:

1. **Experience replay** (Experiment 1): Without replay, DQN trains on correlated sequential transitions, causing noisy Q-values and unstable reward curves. Replay breaks this correlation by sampling random mini-batches from a buffer.

2. **Target network** (Experiment 2): Without a target network, the bootstrapping target shifts with every gradient step, creating a positive feedback loop. The target network freezes the target for C steps, decoupling prediction from target and producing more stable Q-value convergence.

3. **Epsilon schedule** (Experiment 3): Fixed epsilon at either extreme fails -- always random never exploits, always greedy never explores. A decaying schedule (linear or exponential) balances exploration and exploitation, with linear decay being the most robust default.

For the full mathematical treatment and interview Q&A, see [dqn-from-scratch-interview.md](./dqn-from-scratch-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)